In [2]:
import pandas as pd
import numpy as np
from google.colab import files
import re

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name, header=None, engine="openpyxl")
df["ordem"] = df.index

def buscar_valor(campo):
    for i in range(len(df)):
        for j in range(len(df.columns)):
            if str(df.iloc[i,j]).strip() == campo:
                for col in range(j+1,len(df.columns)):
                    v = df.iloc[i,col]
                    if pd.notnull(v): return v
                for row in range(i+1,len(df)):
                    v = df.iloc[row,j]
                    if pd.notnull(v): return v
    return None

unidade_base   = buscar_valor("Unidade")
menu_grid_cod  = buscar_valor("Cod. Menu Grid")
data_custo_raw = buscar_valor("Data de Custo")
companhia      = buscar_valor("Companhia")
menu_grid      = buscar_valor("Menu Grid")

# Extrai apenas a data (formato DD/MM/AAAA) do campo de data de custo
data_custo = None
if data_custo_raw:
    match = re.search(r"\d{2}/\d{2}/\d{4}", str(data_custo_raw))
    if match:
        data_custo = match.group(0)
    else:
        data_custo = str(data_custo_raw).split("-")[0].strip()

is_new_recipe = (df.iloc[:, 0].isnull() | (df.iloc[:, 0].astype(str).str.strip() == "")) & (df.iloc[:, 1].notnull())
df["Grupo_Receita"] = is_new_recipe.cumsum()
df["Nome_Receita_Pai"] = df.iloc[:, 2].where(is_new_recipe)
df["Agrupador_Receita"] = df.groupby("Grupo_Receita")["Nome_Receita_Pai"].transform("first")

df["Rendimento_Liq"] = pd.NA
df["Rendimento_KG"] = pd.NA
df["Rendimento_Unidade"] = pd.NA
df["Custo_Rendimento"] = pd.NA

# Guardamos um mapeamento de grupo para rendimento líquido para usar nos cálculos dos itens
grupo_to_rend_liq = {}

for g_id, bloco in df.groupby("Grupo_Receita"):
    if g_id == 0: continue
    last_liq, last_kg, last_unit, last_cost = pd.NA, pd.NA, pd.NA, pd.NA
    for idx in bloco.index:
        texto_a = str(bloco.loc[idx, 0]).upper()
        valor_e = pd.to_numeric(bloco.loc[idx, 4], errors="coerce")
        unid_f  = bloco.loc[idx, 5]
        custo_i = bloco.loc[idx, 8]
        if "RENDIMENTO L" in texto_a and pd.notnull(valor_e): last_liq = valor_e
        if ("RENDIMENTO KG" in texto_a or "RENDIMENTO UN" in texto_a) and pd.notnull(valor_e):
            last_kg = valor_e
            if pd.notnull(unid_f): last_unit = unid_f
            if pd.notnull(custo_i): last_cost = custo_i
    header_idx = bloco.index.min()
    # Exibimos apenas na linha do cabeçalho principal no relatório final
    df.loc[header_idx, "Rendimento_Liq"] = last_liq
    df.loc[header_idx, "Rendimento_KG"] = last_kg
    df.loc[header_idx, "Rendimento_Unidade"] = last_unit
    df.loc[header_idx, "Custo_Rendimento"] = last_cost

    # Mas guardamos a referência do Rendimento Líquido para o cálculo interno dos produtos do grupo
    if pd.notnull(last_liq):
        grupo_to_rend_liq[g_id] = last_liq

df["Class"] = df.iloc[:,0].where(df.iloc[:,0].isin(["P","R","S"]))
# Filtramos apenas linhas que representam itens ou headers de receita
df_base = df[df.iloc[:,1].notnull()].copy()

def capturar_dados_mesma_linha(idx):
    res = {"Quantidade": None, "UN": None, "Custo_Unit": None, "Yield": None, "Custo_c_Yield": None}
    linha_atual = df.loc[idx]
    # Coluna 8 = Q (Qtd), Coluna 9 = J (UN), Coluna 10 = K (Custo da Coluna K)
    if pd.notnull(linha_atual[8]) or pd.notnull(linha_atual[10]):
        res["Quantidade"] = linha_atual[8]
        res["UN"] = linha_atual[9]
        res["Custo_Unit"] = linha_atual[10]
        res["Yield"] = linha_atual[6]
        res["Custo_c_Yield"] = linha_atual[7]
    return pd.Series(res)

df_resultado = df.apply(lambda x: capturar_dados_mesma_linha(x.name), axis=1, result_type="expand")
df_base = df_base.join(df_resultado.loc[df_base.index])

df_base["qtd_num"] = pd.to_numeric(df_base["Quantidade"], errors="coerce")
df_base["rend_liq_num"] = pd.to_numeric(df_base["Rendimento_Liq"], errors="coerce")
df_base["rend_kg_num"] = pd.to_numeric(df_base["Rendimento_KG"], errors="coerce")

is_header = df_base["Class"].isnull()

# Para cálculo da Qtd Unt e do Custo Unitário nos itens, precisamos mapear o Rendimento Líquido do grupo correspondente
df_base["grupo_rend_liq"] = df_base["Grupo_Receita"].map(grupo_to_rend_liq).astype(float)

# Calculo da Qtd Unt apenas para ingredientes
df_base["Qtd_Unt_Calculada"] = np.where(
    ~is_header,
    (df_base["qtd_num"] / df_base["grupo_rend_liq"].replace(0, np.nan)) * df_base["rend_kg_num"].astype(float),
    np.nan
)

# Quantidade deve ser nula para Header conforme solicitado
df_base["Quantidade_Exibicao"] = np.where(is_header, np.nan, df_base["qtd_num"])

# Consolidate 'Custo_Unit' and 'Custo_Rendimento' into a new 'Custo_Sem_Yield' column
df_base["Custo_Sem_Yield"] = df_base["Custo_Unit"].fillna(df_base["Custo_Rendimento"])
df_base["Custo_Sem_Yield_num"] = pd.to_numeric(df_base["Custo_Sem_Yield"], errors="coerce")

# Calculo do Custo Unitário:
# Se for Header (is_header é True): traz o custo original sem alteração
# Se for ingrediente (~is_header): divide o custo do produto pelo rendimento líquido da receita (grupo)
df_base["Custo_Unitario_Calculado"] = np.where(
    is_header,
    df_base["Custo_Sem_Yield_num"],
    df_base["Custo_Sem_Yield_num"] / df_base["grupo_rend_liq"].replace(0, np.nan)
)

df_final = pd.DataFrame({
    "Unidade": unidade_base,
    "Cod Menu Grid": menu_grid_cod,
    "Data Custo": data_custo,
    "Companhia": companhia,
    "Menu Grid": menu_grid,
    "Class": df_base["Class"],
    "Código": df_base.iloc[:,1],
    "Receita Principal": df_base["Agrupador_Receita"],
    "Produto/Receita": df_base.iloc[:,2],
    "Yield": df_base["Yield"],
    "Quantidade": df_base["Quantidade_Exibicao"],
    "Qtd Unt": df_base["Qtd_Unt_Calculada"],
    "UN": df_base["UN"],
    "Rendimento_Liq": df_base["Rendimento_Liq"],
    "Rendimento_KG": df_base["Rendimento_KG"],
    "Unidade Rendimento": df_base["Rendimento_Unidade"],
    "Custo": df_base["Custo_Sem_Yield"],
    "Custo Unitário": df_base["Custo_Unitario_Calculado"]
})

# Remover linhas inuteis
has_trash = df_final["Produto/Receita"].astype(str).str.contains("Inflação|Total|Rendimento|Histórico", na=False)
df_final = df_final[~(has_trash & df_final["Class"].notnull())]
df_final = df_final[df_final["Código"].astype(str).str.upper() != "CÓDIGO"].reset_index(drop=True)

output = "dados_receita.xlsx"
df_final.to_excel(output, index=False)
files.download(output)

Saving RECEITA 20.03.26 LATAM GRU.xlsx to RECEITA 20.03.26 LATAM GRU (1).xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>